In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../assignments/DATA_OFFER_ALLOCATION.csv")

### 1. Data Split

In [ ]:
mask_made_purchase = df['POINTS_PURCHASED'] > 0

SPLIT_TRAIN = 0.8
SPLIT_VAL   = 0.1   # remaining 10% goes to test

duplicates = df.loc[mask_made_purchase, 'MEMBER_KEY']
# keep=False marks ALL occurrences of a key that appears more than once
MEMBER_KEY_WITH_MULTIPLE_PURCHASES = np.array(duplicates[duplicates.duplicated(keep=False)].unique())
MEMBER_KEY_WITH_SINGLE_PURCHASE    = np.array(duplicates[~duplicates.duplicated(keep=False)].unique())

print(f"Total purchase rows    : {mask_made_purchase.sum()}")
print(f"Multi-purchase members : {len(MEMBER_KEY_WITH_MULTIPLE_PURCHASES)}")
print(f"Single-purchase members: {len(MEMBER_KEY_WITH_SINGLE_PURCHASE)}")

In [ ]:
rng = np.random.default_rng(42)

# Shuffle both groups independently
multi_shuffled  = rng.permutation(MEMBER_KEY_WITH_MULTIPLE_PURCHASES)
single_shuffled = rng.permutation(MEMBER_KEY_WITH_SINGLE_PURCHASE)

# Compute split sizes (proportional within each group)
n_multi  = len(multi_shuffled)
n_single = len(single_shuffled)

n_multi_train  = int(SPLIT_TRAIN * n_multi)
n_multi_val    = int(SPLIT_VAL   * n_multi)

n_single_train = int(SPLIT_TRAIN * n_single)
n_single_val   = int(SPLIT_VAL   * n_single)

# Slice member keys per split
multi_train_keys  = multi_shuffled[:n_multi_train]
multi_val_keys    = multi_shuffled[n_multi_train : n_multi_train + n_multi_val]
multi_test_keys   = multi_shuffled[n_multi_train + n_multi_val:]

single_train_keys = single_shuffled[:n_single_train]
single_val_keys   = single_shuffled[n_single_train : n_single_train + n_single_val]
single_test_keys  = single_shuffled[n_single_train + n_single_val:]

# Combine multi + single for each split
train_members = np.concatenate([multi_train_keys,  single_train_keys])
val_members   = np.concatenate([multi_val_keys,    single_val_keys])
test_members  = np.concatenate([multi_test_keys,   single_test_keys])

# Build dataframes — all rows whose MEMBER_KEY belongs to the split
train_df = df[(df['MEMBER_KEY'].isin(train_members)) & mask_made_purchase].reset_index(drop=True)
val_df   = df[(df['MEMBER_KEY'].isin(val_members)) & mask_made_purchase].reset_index(drop=True)
test_df  = df[(df['MEMBER_KEY'].isin(test_members)) & mask_made_purchase].reset_index(drop=True)

print(f"{'Split':<8} {'rows':>6}  {'members':>8}  {'multi':>6}  {'single':>7}")
print("-" * 45)
print(f"{'Train':<8} {len(train_df):>6}  {len(train_members):>8}  {len(multi_train_keys):>6}  {len(single_train_keys):>7}")
print(f"{'Val':<8} {len(val_df):>6}  {len(val_members):>8}  {len(multi_val_keys):>6}  {len(single_val_keys):>7}")
print(f"{'Test':<8} {len(test_df):>6}  {len(test_members):>8}  {len(multi_test_keys):>6}  {len(single_test_keys):>7}")

In [ ]:
# Verify no member key appears in more than one split
train_set = set(train_df['MEMBER_KEY'])
val_set   = set(val_df['MEMBER_KEY'])
test_set  = set(test_df['MEMBER_KEY'])

assert train_set.isdisjoint(val_set),  "Overlap between train and val!"
assert train_set.isdisjoint(test_set), "Overlap between train and test!"
assert val_set.isdisjoint(test_set),   "Overlap between val and test!"

print("No member overlap between splits — verified!")

### 2. Feature Transformations

In [ ]:
COLS_TO_DROP = [
    'SESSION_KEY',
    'MEMBER_KEY',
    'OFFER_RICHNESS_APPLIED',
    'PRICE_PER_POINT',
    'OFFER_RICHNESS_APPLIED_ON_LAST_PURCHASE_L12M',
    'POINTS_PURCHASED_LAST_TRANX_L12M',
    # 'FLAG_TRANSACTION'
]


def extract_session_date_features(df):
    df = df.copy()
    dt = pd.to_datetime(df['SESSION_DATE'])
    df['HOUR_OF_DAY'] = dt.dt.hour
    df['DAY_OF_WEEK']  = dt.dt.dayofweek   # 0=Monday, 6=Sunday
    return df.drop(columns=['SESSION_DATE'])


def transform_current_balance(df):
    """log1p on CURRENT_BALANCE + binary flag for zero-balance members."""
    df = df.copy()
    col = 'CURRENT_BALANCE'
    mask_sentinel = df[col] == 0
    df['BALANCE_ZERO'] = (df['CURRENT_BALANCE'] == 0).astype(int)
    df[col] = np.log1p(df[col])
    df.loc[mask_sentinel, col] = 0
    return df


def bucket_count_tranx(df):
    """Bucket COUNT_TRANX_L12M: 0→0, 1→1, 2→2, 3-5→3, 6-9→4, 10+→5."""
    df = df.copy()
    bins   = [-1, 0, 1, 2, 5, 9, np.inf]
    labels = [0, 1, 2, 3, 4, 5]
    df['COUNT_TRANX_L12M'] = (
        pd.cut(df['COUNT_TRANX_L12M'], bins=bins, labels=labels).astype(int)
    )
    return df


def transform_offer_richness_vs_last(df):
    """
    CAT_OFFER_SERVED_WORSE_THAN_LAST:
      0 = current offer same or better than last purchase offer
      1 = member never purchased in L12M (NaN last offer)
      2 = current offer worse than last purchase offer
    Drops LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M afterward.
    """
    df = df.copy()
    last    = df['LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M']
    current = df['OFFER_RICHNESS_SERVED']
    df['CAT_OFFER_SERVED_WORSE_THAN_LAST'] = np.where(
        last.isna(), 1,
        np.where(current >= last, 0, 2)
    ).astype(int)
    return df.drop(columns=['LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M'])


def transform_days_since_last_purchase(df):
    """Missing flag + sqrt on DAYS_SINCE_LAST_PURCHASE_L12M. Sentinel = 0."""
    df = df.copy()
    col = 'DAYS_SINCE_LAST_PURCHASE_L12M'
    mask_sentinel = df[col] == 9999
    # df['missing_DAYS_SINCE_LAST_PURCHASE_L12M'] = df[col].isna().astype(int)
    df['missing_DAYS_SINCE_LAST_PURCHASE_L12M'] = mask_sentinel.astype(int)
    df[col] = np.sqrt(df[col]).fillna(0)
    df.loc[mask_sentinel, col] = 0
    return df


def transform_days_since_last_visit(df):
    """Missing flag + log on DAYS_SINCE_LAST_VISIT_NO_PURCHASE. Sentinel = 0."""
    df = df.copy()
    col = 'DAYS_SINCE_LAST_VISIT_NO_PURCHASE'
    mask_sentinel = df[col] == 9999
    # df['missing_DAYS_SINCE_LAST_VISIT_NO_PURCHASE'] = df[col].isna().astype(int)
    df['missing_DAYS_SINCE_LAST_VISIT_NO_PURCHASE'] = mask_sentinel.astype(int)
    df[col] = np.log1p(df[col]).fillna(0)
    df.loc[mask_sentinel, col] = 0
    return df


def transform_points_purchased(df):
    """Log for points purchased"""
    df = df.copy()
    col = 'POINTS_PURCHASED'
    df[col] = np.log1p(df[col]).fillna(0)
    return df


def drop_columns(df):
    return df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])


def apply_feature_transformations(df):
    df = extract_session_date_features(df)
    df = transform_current_balance(df)
    df = bucket_count_tranx(df)
    df = transform_offer_richness_vs_last(df)
    df = transform_days_since_last_purchase(df)
    df = transform_days_since_last_visit(df)
    # df = transform_points_purchased(df) # transformation on target (doesn't perform well)
    df = drop_columns(df)
    return df

In [ ]:
train_df_transformed = apply_feature_transformations(train_df)
val_df_transformed   = apply_feature_transformations(val_df)
test_df_transformed  = apply_feature_transformations(test_df)

print("Columns after transformation:")
print(train_df_transformed.columns.tolist())
train_df_transformed.head()

In [ ]:
test_df_transformed['OFFER_RICHNESS_SERVED'].value_counts(normalize=True) # note: not same distribution as overall

### 3. Modeling — M2: Points Purchased

In [ ]:
import optuna
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from optuna.samplers import TPESampler
import warnings

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

TARGET   = 'POINTS_PURCHASED'
DROP_IN_X = [TARGET]
FEATURES = [c for c in train_df_transformed.columns if c not in DROP_IN_X]

X_train, y_train = train_df_transformed[FEATURES].values, train_df_transformed[TARGET].values
X_val,   y_val   = val_df_transformed[FEATURES].values,   val_df_transformed[TARGET].values
X_test,  y_test  = test_df_transformed[FEATURES].values,  test_df_transformed[TARGET].values

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

In [ ]:
N_TRIALS = 50

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate(model, X, y):
    pred = model.predict(X)
    return {
        'RMSE': rmse(y, pred),
        'MAE':  mean_absolute_error(y, pred),
        'R2':   r2_score(y, pred),
    }


def objective_lasso(trial):
    model = Lasso(
        alpha    = trial.suggest_float('alpha', 1e-4, 100.0, log=True),
        max_iter = 10000,
    )
    model.fit(X_train, y_train)
    return rmse(y_val, model.predict(X_val))


def objective_rf(trial):
    model = RandomForestRegressor(
        n_estimators      = trial.suggest_int('n_estimators', 50, 500),
        max_depth         = trial.suggest_int('max_depth', 3, 20),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 10),
        max_features      = trial.suggest_float('max_features', 0.3, 1.0),
        random_state      = 42,
        n_jobs            = -1,
    )
    model.fit(X_train, y_train)
    return rmse(y_val, model.predict(X_val))


def objective_xgb(trial):
    # Note: objective='reg:squarederror' is bad for tail
    model = xgb.XGBRegressor(
        objective        = 'reg:pseudohubererror',
        n_estimators     = trial.suggest_int('n_estimators', 50, 500),
        max_depth        = trial.suggest_int('max_depth', 3, 10),
        learning_rate    = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
        random_state     = 42,
        n_jobs           = -1,
        verbosity        = 0,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return rmse(y_val, model.predict(X_val))


def objective_lgb(trial):
    # lgb.LGBMRegressor(objective='quantile', alpha=0.9)
    # lgb.LGBMRegressor(objective='tweedie', tweedie_variance_power=1.5)
    model = lgb.LGBMRegressor(
        objective         = 'quantile',
        n_estimators      = trial.suggest_int('n_estimators', 50, 500),
        max_depth         = trial.suggest_int('max_depth', 3, 10),
        learning_rate     = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 20, 200),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = -1,
    )
    model.fit(X_train, y_train)
    return rmse(y_val, model.predict(X_val))

In [ ]:
OBJECTIVES = {
    'Lasso':        objective_lasso,
    'RandomForest': objective_rf,
    'XGBoost':      objective_xgb,
    'LightGBM':     objective_lgb,
}

studies = {}
for name, objective in OBJECTIVES.items():
    print(f"Tuning {name} ({N_TRIALS} trials)...")
    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    studies[name] = study
    print(f"  best val RMSE : {study.best_value:.4f}")
    print(f"  best params   : {study.best_params}\n")

In [ ]:
def retrain_best(name, params):
    """Retrain on train+val with the best hyperparameters found."""
    X_tv = np.vstack([X_train, X_val])
    y_tv = np.concatenate([y_train, y_val])

    if name == 'Lasso':
        model = Lasso(**params, max_iter=10000)
    elif name == 'RandomForest':
        model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    elif name == 'XGBoost':
        model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1, verbosity=0)
    elif name == 'LightGBM':
        model = lgb.LGBMRegressor(**params, random_state=42, n_jobs=-1, verbosity=-1)

    model.fit(X_tv, y_tv)
    return model


best_models = {}
records = []
for name, study in studies.items():
    model = retrain_best(name, study.best_params)
    best_models[name] = model
    metrics = evaluate(model, X_test, y_test)
    records.append({'Model': name, **metrics})

results_df = pd.DataFrame(records).set_index('Model').sort_values('RMSE')
print(results_df.round(4))

### 4. Predictions

In [ ]:
predictions_df = test_df_transformed[FEATURES].copy()
predictions_df['y_true'] = y_test

for name, model in best_models.items():
    predictions_df[f'y_pred_{name}'] = model.predict(X_test).clip(min=0)

predictions_df.head(10)

In [ ]:
# # transforming back (if we log transform POINTS_PURCHASED) [doesn't perform well]

# predictions_df2 = predictions_df.copy()

# predictions_df2['y_true'] = np.exp(predictions_df2['y_true'])
# for name, model in best_models.items():
#     predictions_df2[f'y_pred_{name}'] = np.expm1(model.predict(X_test).clip(min=0))

# predictions_df2.head(10)

### 5. Feature Importances

In [ ]:
from yellowbrick.model_selection import FeatureImportances

print(best_models.keys())

# for model_name in ['RandomForest', 'XGBoost']:
#     viz = FeatureImportances(best_models[model_name])
#     viz.show()

In [ ]:
feature_idx = {i: f for i, f in enumerate(FEATURES)}
plt.bar(x=feature_idx.keys(), height=best_models['RandomForest'].feature_importances_, label=feature_idx.values())
plt.legend()
plt.show()